<a href="https://colab.research.google.com/github/tri25msb43048-eng/rag-thu-tuc-hanh-chinh/blob/main/rag_thu_tuc_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG tra cứu thủ tục hành chính (Llamafile + Qdrant)
Chuyển thể từ lab *Introduction to RAG* (Duke). Chạy lần lượt từng ô từ trên xuống.

## 1. Cài thư viện

In [1]:
!pip install -q qdrant-client sentence-transformers openai pandas

## 2. Tải dữ liệu
Tải tệp `thu_tuc_mau.csv` lên (biểu tượng thư mục bên trái → Upload), hoặc chạy ô dưới để chọn tệp.

In [2]:
import os
if not os.path.exists('thu_tuc_mau.csv'):
    from google.colab import files
    files.upload()

Saving thu_tuc_mau.csv to thu_tuc_mau.csv


## 3. Chạy Llamafile (cổng 8081)
Lần đầu tải model khoảng 1,23 GB, mất vài phút.

In [3]:
import subprocess, time, os
from openai import OpenAI

URL = 'https://huggingface.co/mozilla-ai/llamafile_0.10/resolve/main/Qwen3.5-0.8B-Q8_0.llamafile'
if not os.path.exists('model.llamafile'):
    subprocess.run(['wget', '-q', '-O', 'model.llamafile', URL], check=True)
os.chmod('model.llamafile', 0o755)

# Cổng 8080 trên Colab đã bị dịch vụ nội bộ chiếm, nên dùng 8081.
# Bản 0.10 không có cờ --nobrowser; Colab không có Vulkan nên ép chạy CPU (--gpu disable).
subprocess.run('pkill -f model.llamafile', shell=True)
log = open('llamafile.log', 'w')
proc = subprocess.Popen(
    ['sh', './model.llamafile', '--server', '--gpu', 'disable',
     '--host', '127.0.0.1', '--port', '8081'],
    stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    start_new_session=True)

client = OpenAI(base_url='http://127.0.0.1:8081/v1', api_key='sk-no-key-required')
for i in range(48):  # tối đa 4 phút
    if proc.poll() is not None:
        print(f'Llamafile đã tắt (mã thoát {proc.returncode}). Log:\n')
        print(open('llamafile.log').read()[-3000:]); break
    try:
        r = client.chat.completions.create(model='LLaMA_CPP', max_tokens=20,
                messages=[{'role': 'user', 'content': 'Xin chào'}])
        print('LLM sẵn sàng. Trả lời thử:', r.choices[0].message.content); break
    except Exception:
        print(f'Đang nạp model... {5*(i+1)} giây'); time.sleep(5)

Đang nạp model... 5 giây
LLM sẵn sàng. Trả lời thử: Chào bạn! Rất vui được gặp gỡ. Tôi là AI assistant, có thể hỗ trợ với


In [11]:
import subprocess, time, os, json, urllib.request
from openai import OpenAI

# 1. Tìm và tải tệp chạy llamafile mới nhất (không kèm model) từ GitHub
if not os.path.exists('llamafile_bin') or os.path.getsize('llamafile_bin') < 1_000_000:
    rel = json.load(urllib.request.urlopen(
        'https://api.github.com/repos/mozilla-ai/llamafile/releases/latest'))
    tep = [a for a in rel['assets']
           if a['name'].startswith('llamafile-') and '.' not in a['name'].replace(rel['tag_name'], '')]
    print('Bản phát hành:', rel['tag_name'], '| các tệp:', [a['name'] for a in rel['assets']][:8])
    url = tep[0]['browser_download_url']
    print('Đang tải', url)
    subprocess.run(['wget', '-q', '-O', 'llamafile_bin', url], check=True)
os.chmod('llamafile_bin', 0o755)
print('llamafile_bin', round(os.path.getsize('llamafile_bin') / 1e6), 'MB')

# 2. Chạy model 3B bằng tệp chạy riêng
subprocess.run('pkill -f llamafile', shell=True); time.sleep(2)
log = open('llamafile.log', 'w')
proc = subprocess.Popen(
    ['sh', './llamafile_bin', '-m', 'qwen3b.gguf', '-t', str(os.cpu_count()),
     '--server', '--gpu', 'disable', '--host', '127.0.0.1', '--port', '8081'],
    stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    start_new_session=True)

client = OpenAI(base_url='http://127.0.0.1:8081/v1', api_key='sk-no-key-required')
for i in range(60):
    if proc.poll() is not None:
        print(f'Llamafile đã tắt (mã thoát {proc.returncode}). Log:\n')
        print(open('llamafile.log').read()[-3000:]); break
    try:
        r = client.chat.completions.create(model='LLaMA_CPP', max_tokens=30,
                messages=[{'role': 'user', 'content': 'Bạn là model gì?'}])
        print('LLM sẵn sàng. Trả lời thử:', r.choices[0].message.content); break
    except Exception:
        print(f'Đang nạp model... {5*(i+1)} giây'); time.sleep(5)

Bản phát hành: 0.10.6 | các tệp: ['diffusionfile-0.10.6', 'llamafile-0.10.6', 'llamafile-0.10.6-thin', 'llamafile-0.10.6.zip', 'transcribefile-0.10.6', 'whisperfile-0.10.6', 'zipalign-0.10.6']
Đang tải https://github.com/mozilla-ai/llamafile/releases/download/0.10.6/llamafile-0.10.6
llamafile_bin 368 MB
Đang nạp model... 5 giây
LLM sẵn sàng. Trả lời thử: Xin lỗi, nhưng tôi không thể xác định mình là model của bất kỳ ai. Tôi là một trợ lý AI được训练由Alibaba Cloud


## 4. Đọc dữ liệu và tạo kho vector

In [13]:
import pandas as pd
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

TEN_COLLECTION = 'thu_tuc_hanh_chinh'
NGUONG_DIEM = 0.35

df = pd.read_csv('thu_tuc_mau.csv').fillna('')
tu_khoa = {
    'VD-01': 'làm giấy khai sinh; khai sinh cho con; đăng ký cho em bé',
    'VD-05': 'sao y; sao y bản chính; photo công chứng; chứng thực căn cước; công chứng bằng cấp',
    'VD-06': 'giấy xác nhận độc thân; xác nhận chưa kết hôn',
}
df['tu_khoa'] = df['ma'].map(tu_khoa).fillna('')
thu_tuc = df.to_dict('records')

def van_ban_de_tim(tt):
    return f"{tt['ten']}. {tt['mo_ta']}. {tt['tu_khoa']}. Lĩnh vực {tt['linh_vuc']}."

encoder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
qdrant = QdrantClient(':memory:')
if qdrant.collection_exists(TEN_COLLECTION):
    qdrant.delete_collection(TEN_COLLECTION)
qdrant.create_collection(
    collection_name=TEN_COLLECTION,
    vectors_config=models.VectorParams(size=encoder.get_sentence_embedding_dimension(),
                                       distance=models.Distance.COSINE),
)
vectors = encoder.encode([van_ban_de_tim(tt) for tt in thu_tuc])
qdrant.upload_points(
    collection_name=TEN_COLLECTION,
    points=[models.PointStruct(id=i, vector=v.tolist(), payload=tt)
            for i, (v, tt) in enumerate(zip(vectors, thu_tuc))],
)
print('Đã nạp', len(thu_tuc), 'thủ tục')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_891/2491904926.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vectors_config=models.VectorParams(size=encoder.get_sentence_embedding_dimension(),


Đã nạp 6 thủ tục


## 5. Tìm kiếm

In [14]:
def tim_thu_tuc(cau_hoi, k=3):
    pts = qdrant.query_points(collection_name=TEN_COLLECTION,
                              query=encoder.encode(cau_hoi).tolist(), limit=k).points
    return [(p.payload, p.score) for p in pts if p.score >= NGUONG_DIEM]

for tt, diem in tim_thu_tuc('Làm giấy khai sinh cho con cần mang gì?'):
    print(tt['ma'], tt['ten'], round(diem, 2))

VD-01 Đăng ký khai sinh 0.6
VD-04 Cấp bản sao trích lục hộ tịch 0.58
VD-03 Đăng ký khai tử 0.48


In [20]:
# 1. Kiểm tra bước tìm kiếm hiện tại
print('--- Điểm cosine hiện tại cho "sao y căn cước ở đâu" ---')
for p in qdrant.query_points(collection_name=TEN_COLLECTION,
        query=encoder.encode('sao y căn cước ở đâu').tolist(), limit=3).points:
    print(' ', p.payload['ma'], p.payload['ten'], round(p.score, 2))

# 2. Từ khóa cách nói của người dân (cán bộ Một cửa bổ sung dần)
TU_KHOA = {
    'VD-01': ['làm giấy khai sinh', 'khai sinh cho con', 'đăng ký cho em bé'],
    'VD-05': ['sao y', 'photo công chứng', 'chứng thực', 'công chứng'],
    'VD-06': ['xác nhận độc thân', 'xác nhận chưa kết hôn'],
}
THEO_MA = {t['ma']: t for t in thu_tuc}

# 3. Tìm kiếm lai: khớp từ khóa trước, rồi mới đến điểm cosine
def tim_thu_tuc(cau_hoi, k=3):
    q = cau_hoi.lower()
    khop = [(THEO_MA[ma], 1.0) for ma, ds in TU_KHOA.items()
            if ma in THEO_MA and any(tk in q for tk in ds)]
    da_co = {t['ma'] for t, _ in khop}
    pts = qdrant.query_points(collection_name=TEN_COLLECTION,
                              query=encoder.encode(cau_hoi).tolist(), limit=k).points
    theo_nghia = [(p.payload, p.score) for p in pts
                  if p.score >= NGUONG_DIEM and p.payload['ma'] not in da_co]
    return (khop + theo_nghia)[:k]

print('\n--- Sau khi sửa ---')
for t, d in tim_thu_tuc('sao y căn cước ở đâu'):
    print(' ', t['ma'], t['ten'], round(d, 2), '(khớp từ khóa)' if d == 1.0 else '')
print('\nCó bước kiểm tra liên quan:', 'kiem_tra_lien_quan' in globals())

--- Điểm cosine hiện tại cho "sao y căn cước ở đâu" ---
  VD-05 Chứng thực bản sao từ bản chính 0.17
  VD-04 Cấp bản sao trích lục hộ tịch 0.12
  VD-01 Đăng ký khai sinh 0.07

--- Sau khi sửa ---
  VD-05 Chứng thực bản sao từ bản chính 1.0 (khớp từ khóa)

Có bước kiểm tra liên quan: False


## 6. Gọi LLM với tài liệu tìm được

In [24]:
from openai import OpenAI
client = OpenAI(base_url='http://127.0.0.1:8081/v1', api_key='sk-no-key-required')

def kiem_tra_lien_quan(cau_hoi, tt):
    r = client.chat.completions.create(
        model='LLaMA_CPP', temperature=0, max_tokens=5,
        messages=[{'role': 'user', 'content':
            f'Người dân hỏi: "{cau_hoi}"\n'
            f"Thủ tục: {tt['ten']} — {tt['mo_ta']}\n"
            'Thủ tục này có đúng là việc người dân muốn làm không? '
            'Chỉ trả lời một từ: CO hoặc KHONG.'}])
    return r.choices[0].message.content.strip().upper().startswith('CO')

def tra_loi(cau_hoi):
    ds = [x for x in tim_thu_tuc(cau_hoi) if kiem_tra_lien_quan(cau_hoi, x[0])]
    if not ds:
        return 'Chưa tìm thấy thủ tục phù hợp. Vui lòng liên hệ Bộ phận Một cửa UBND xã.'
    khoi = []
    for t, _ in ds:
        khoi.append(
            f"Thủ tục: {t['ten']} [{t['ma']}]\n"
            f"- Hồ sơ: {t['ho_so']}\n"
            f"- Thời hạn: {t['thoi_han']}\n"
            f"- Lệ phí: {t['le_phi']}\n"
            f"- Nơi nộp: {t['noi_nop']}")
    mo_dau = ('Anh/chị có thể đang hỏi về thủ tục sau:' if len(ds) == 1
              else 'Có thể anh/chị đang hỏi về một trong các thủ tục sau, vui lòng xác nhận:')
    return mo_dau + '\n\n' + '\n\n'.join(khoi)

## 7. Chạy thử

In [25]:
for q in ['Làm giấy khai sinh cho con cần mang gì?',
          'Tôi muốn xin giấy xác nhận độc thân',
          'sao y căn cước ở đâu',
          'xin cấp giấy phép xây dựng nhà ở']:
    print('Người dân:', q)
    print('Trợ lý:', tra_loi(q), '\n')

Người dân: Làm giấy khai sinh cho con cần mang gì?
Trợ lý: Anh/chị có thể đang hỏi về thủ tục sau:

Thủ tục: Đăng ký khai sinh [VD-01]
- Hồ sơ: Tờ khai đăng ký khai sinh; Giấy chứng sinh hoặc giấy tờ thay thế; Giấy tờ tùy thân của người đi đăng ký
- Thời hạn: Trong ngày làm việc nếu nộp hồ sơ đầy đủ
- Lệ phí: Miễn lệ phí khi đăng ký đúng hạn
- Nơi nộp: Bộ phận Một cửa UBND xã hoặc Cổng Dịch vụ công 

Người dân: Tôi muốn xin giấy xác nhận độc thân
Trợ lý: Anh/chị có thể đang hỏi về thủ tục sau:

Thủ tục: Xác nhận tình trạng hôn nhân [VD-06]
- Hồ sơ: Tờ khai cấp giấy xác nhận tình trạng hôn nhân; Giấy tờ tùy thân
- Thời hạn: Trong 3 ngày làm việc
- Lệ phí: Theo mức thu của tỉnh
- Nơi nộp: Bộ phận Một cửa UBND xã hoặc Cổng Dịch vụ công 

Người dân: sao y căn cước ở đâu
Trợ lý: Anh/chị có thể đang hỏi về thủ tục sau:

Thủ tục: Chứng thực bản sao từ bản chính [VD-05]
- Hồ sơ: Bản chính giấy tờ; Bản sao cần chứng thực
- Thời hạn: Trong ngày làm việc
- Lệ phí: Theo số trang chứng thực
- Nơi n